# Notebook 04 – Matrix Factorization

- From-scratch MF demonstration (NumPy SGD)
- SurpriseMF for full experiments
- Ablation: vary latent dimension d
- Ablation: vary regularisation λ
- Top-K ranking metrics
- Qualitative recommendations


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.matrix_factorization import (
    MatrixFactorization, SurpriseMF,
    mf_sensitivity, mf_lambda_sensitivity,
    get_topk_recs_mf
)
from src.evaluation import rating_metrics, topk_metrics

sns.set_style('whitegrid')
os.makedirs('../results', exist_ok=True)
SEED = 42
np.random.seed(SEED)

In [ ]:
train  = pd.read_parquet('../data/train.parquet')
test   = pd.read_parquet('../data/test.parquet')
movies = pd.read_csv('../data/movie.csv')

n_users  = int(train['user_idx'].max()) + 1
n_movies = int(train['movie_idx'].max()) + 1
print(f'Train: {len(train):,}   Test: {len(test):,}')
print(f'n_users={n_users:,}  n_movies={n_movies:,}')

## 4.1 From-scratch MF (educational demonstration)

Run on a small subset to demonstrate the SGD algorithm.
Use SurpriseMF (Section 4.2+) for actual experiments.

In [ ]:
small_train = train.sample(frac=0.05, random_state=SEED)
small_test  = test.sample(frac=0.05,  random_state=SEED)

mf_scratch = MatrixFactorization(
    n_users=n_users, n_items=n_movies,
    d=20, lam=0.02, lr=0.005, n_epochs=15
)
mf_scratch.fit(small_train)
scratch_metrics = mf_scratch.evaluate(small_test, label='MF-scratch (5% data)')
print(scratch_metrics)

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(mf_scratch.train_rmse_history)+1),
         mf_scratch.train_rmse_history, marker='o', color='steelblue')
plt.title('MF (from scratch) – Training RMSE per Epoch')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.tight_layout()
plt.savefig('../results/mf_scratch_train_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.2 Ablation – vary latent dimension d

Uses 50% subsample + 30 epochs so larger d has a fair chance to converge.

In [ ]:
# Load from saved CSV if already run via run_all.py (recommended)
df_d = pd.read_csv('../results/mf_d_sensitivity.csv', index_col='d')
print(df_d)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_d['RMSE'].plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('MF – Latent Dimension d vs RMSE')
axes[0].set_xlabel('d (latent factors)')
axes[0].set_ylabel('RMSE')

df_d['MAE'].plot(ax=axes[1], marker='s', color='coral')
axes[1].set_title('MF – Latent Dimension d vs MAE')
axes[1].set_xlabel('d (latent factors)')
axes[1].set_ylabel('MAE')

plt.tight_layout()
plt.savefig('../results/mf_d_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.3 Ablation – vary regularisation λ

In [ ]:
df_lam = pd.read_csv('../results/mf_lambda_sensitivity.csv', index_col='lambda')
print(df_lam)

df_lam['RMSE'].plot(marker='o', logx=True, figsize=(7, 4), color='steelblue')
plt.title('MF – Regularisation λ vs RMSE')
plt.xlabel('λ (log scale)')
plt.ylabel('RMSE')
plt.tight_layout()
plt.savefig('../results/mf_lambda_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Train final MF with best hyperparameters

In [ ]:
best_d   = int(df_d['RMSE'].idxmin())
best_lam = float(df_lam['RMSE'].idxmin())
print(f'Best d={best_d}  best lambda={best_lam}')

mf = SurpriseMF(d=best_d, n_epochs=20, lr_all=0.005, reg_all=best_lam)
mf.fit(train)
mf_metrics = mf.evaluate(test, label=f'MF d={best_d}')
print(mf_metrics)

## 4.5 Top-K Ranking Metrics

In [ ]:
sample_users = test['user_idx'].unique()[:500]
test_sample  = test[test['user_idx'].isin(sample_users)].copy()

test_sample['pred_rating'] = mf.predict_df(test_sample)
test_sample = test_sample.rename(columns={'rating': 'true_rating'})

print(f'Top-10 metrics for MF (d={best_d}):')
topk_mf = topk_metrics(test_sample, k=10, threshold=4.0)
print(topk_mf)

## 4.6 Qualitative Recommendations

In [ ]:
sample_uid   = train['str_userId'].iloc[0]
rated_str    = set(train[train['str_userId'] == sample_uid]['str_movieId'])
all_str_mids = train['str_movieId'].unique()

top10 = get_topk_recs_mf(mf, sample_uid, all_str_mids, rated_str, k=10)

print(f'Top-10 recommendations for user {sample_uid} '
      f'(rated {len(rated_str)} movies):\n')
for rank, (mid, score) in enumerate(top10, 1):
    t     = movies[movies['movieId'] == int(mid)]['title'].values
    title = t[0] if len(t) > 0 else mid
    print(f'  {rank:>2}. {title:<55} predicted={score:.2f}')

In [ ]:
pd.DataFrame([mf_metrics]).to_csv('../results/mf_results.csv', index=False)
pd.DataFrame([topk_mf]).to_csv('../results/mf_topk_metrics.csv', index=False)
print('Saved.')